[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/04_autograd_unmasked.ipynb)

# ⚒️ Act I · Quest 04 — Autograd Unmasked

*PyTorch's engine, side-by-side with the one YOU forged. Same machine, industrial scale.*

⬅️ [03_tensors_the_real_metal](03_tensors_the_real_metal.ipynb)  •  [05_your_first_real_network](05_your_first_real_network.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## The reveal

In Quest 02 you built `Value`: numbers that remember their history and backpropagate gradients.
PyTorch tensors do **exactly this** when you set `requires_grad=True`. Let's prove it — same
expression, your engine vs theirs.

In [ ]:
class Value:
    """A scalar that remembers how it was made — and can compute gradients."""
    def __init__(self, data, _parents=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._parents = _parents
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad          # d(a+b)/da = 1
            other.grad += out.grad         # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t ** 2) * out.grad  # d tanh/dx = 1 - tanh^2
        out._backward = _backward
        return out

    def backward(self):
        # Topological sort: visit children before parents, then run the
        # chain rule backwards through the whole graph.
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    # conveniences so expressions read naturally
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __repr__(self): return f"Value({self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
# ---- YOUR engine ----------------------------------------------------------
a1, b1, c1 = Value(2.0), Value(-3.0), Value(10.0)
f1 = (a1 * b1 + c1).tanh()
f1.backward()

# ---- PyTorch --------------------------------------------------------------
a2 = torch.tensor(2.0, requires_grad=True)
b2 = torch.tensor(-3.0, requires_grad=True)
c2 = torch.tensor(10.0, requires_grad=True)
f2 = (a2 * b2 + c2).tanh()
f2.backward()

print(f"{'':10s}{'your engine':>14s}{'pytorch':>12s}")
for name, mine, theirs in [("df/da", a1.grad, a2.grad), ("df/db", b1.grad, b2.grad), ("df/dc", c1.grad, c2.grad)]:
    print(f"{name:10s}{mine:14.6f}{theirs.item():12.6f}")
print("\nIdentical. You already understand PyTorch's core.")

### Peeking at the graph

Your `Value` stored `_parents` and `_op`. PyTorch stores the same thing in `grad_fn` — every
tensor knows the operation that created it.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = y + 5
w = z.sin()

node = w.grad_fn
print("walking the graph backwards from w:")
while node is not None:
    print("  ", type(node).__name__)
    node = node.next_functions[0][0] if node.next_functions else None

### The three rules of living with autograd

**Rule 1 — gradients accumulate.** `backward()` *adds* into `.grad` (your engine did too:
`self.grad += ...`). Zero them between steps or they pile up.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"after backward #{i+1}: x.grad = {x.grad.item()}  (true dy/dx is 4 — it's stacking!)")
x.grad.zero_()
print("after zero_():", x.grad.item())

**Rule 2 — only leaf tensors keep `.grad`.** A tensor you *created* with `requires_grad=True`
is a leaf. A tensor *computed from* others is not — its gradient flows through but isn't stored.

In [ ]:
w = torch.randn(3, requires_grad=True)      # leaf ✓
scaled = w * 2                               # NOT a leaf — it was computed
loss = (scaled ** 2).sum()
loss.backward()
print("w.grad:", w.grad)
print("scaled.grad:", scaled.grad, " <- None (with a warning if you try in some versions)")
print("\n⚠️ classic bug: `w = torch.randn(3, requires_grad=True) * 0.1` makes w a NON-leaf!")
print("   fix: w = (torch.randn(3) * 0.1).requires_grad_(True)")

**Rule 3 — turn tracking off when you don't need it.** Inference and weight updates shouldn't
build graphs: `torch.no_grad()` for blocks, `.detach()` for single tensors. Saves memory & time.

In [ ]:
w = torch.tensor(5.0, requires_grad=True)
with torch.no_grad():
    pred = w * 3                    # no graph built here
print("inside no_grad:", pred.requires_grad)

frozen = (w * 3).detach()           # snapshot, cut from the graph
print("detached:", frozen.requires_grad)

### Full circle: gradient descent, torch edition

In [ ]:
# Minimize the Rosenbrock banana function — a classic torture test for optimizers
def rosenbrock(x, y):
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

x = torch.tensor(-1.5, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
path = []
for step in range(4000):
    loss = rosenbrock(x, y)
    loss.backward()
    with torch.no_grad():
        x -= 1e-3 * x.grad
        y -= 1e-3 * y.grad
        x.grad.zero_(); y.grad.zero_()
    path.append((x.item(), y.item()))

xs = torch.linspace(-2, 2, 200); ys = torch.linspace(-1, 3, 200)
XX, YY = torch.meshgrid(xs, ys, indexing="xy")
plt.contourf(XX, YY, rosenbrock(XX, YY).log(), levels=30, cmap="viridis")
px, py = zip(*path[::50])
plt.plot(px, py, "r.-", ms=3, lw=1, label="descent path")
plt.plot(1, 1, "w*", ms=15, label="true minimum (1,1)")
plt.legend(); plt.title("Autograd navigating the banana valley"); plt.show()
print(f"reached ({x.item():.3f}, {y.item():.3f}) — target (1, 1)")

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda g: abs(g - 7.0) < 1e-5,
    "y = x**2 + 3x at x=2: dy/dx = 2x + 3 = 7. Build with requires_grad, backward, submit x.grad.item()")
_register("poly_grad", 15,
    lambda g: torch.is_tensor(g) and torch.allclose(g, torch.tensor([2., 4., 6.])),
    "w = torch.tensor([1.,2.,3.], requires_grad=True); loss = (w**2).sum(); backward; submit w.grad")
_register("leaf_fix", 15,
    lambda w: torch.is_tensor(w) and w.is_leaf and w.requires_grad and w.shape == (4,),
    "create a shape-(4,) leaf tensor scaled by 0.1 that still tracks gradients: (torch.randn(4)*0.1).requires_grad_(True)")
_register("valley", 15,
    lambda xy: abs(xy[0] - 1) < 0.15 and abs(xy[1] - 1) < 0.15,
    "run the Rosenbrock descent longer (or nudge lr) until (x,y) is within 0.15 of (1,1); submit (x.item(), y.item())")

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x
y.backward()
check("warmup", x.grad.item())

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `poly_grad` (15 XP) — gradient of `(w**2).sum()` at `w=[1,2,3]`; submit `w.grad`.
- `leaf_fix` (15 XP) — create a *scaled* leaf tensor that keeps `.grad` (the Rule-2 bug, fixed); submit it.
- `valley` (15 XP) — get the banana-valley descent within `0.15` of `(1, 1)`; submit `(x.item(), y.item())`.

In [ ]:
# ⚔️ your attempts here...

# xp_report()